## Simple GenAI app with langchain

In [5]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [6]:
docs = WebBaseLoader("https://en.wikipedia.org/wiki/Generative_AI").load()
docs

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Generative_AI', 'title': 'Generative AI - Wikipedia', 'language': 'en'}, page_content='\n\n\n\nGenerative AI - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\n\nDonate\n\n\nCreate account\n\n\nLog in\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nContents\nmove to sidebar\nhide\n\n\n\n\n(Top)\n\n\n\n\n\n1\nHistory\n\n\n\n\nToggle History subsection\n\n\n\n\n\n1.1\n

In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
documents = text_splitter.split_documents(docs)
documents

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Generative_AI', 'title': 'Generative AI - Wikipedia', 'language': 'en'}, page_content='Generative AI - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\n\nDonate\n\n\nCreate account\n\n\nLog in\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nContents\nmove to sidebar\nhide\n\n\n\n\n(Top)\n\n\n\n\n\n1\nHistory\n\n\n\n\nToggle History subsection\n\n\n\n\n\n1.1\nEarly hi

In [9]:
embeddings = OpenAIEmbeddings(model = "text-embedding-3-large")

In [10]:
vector_db = FAISS.from_documents(documents, embeddings)

In [11]:
vector_db

In [12]:
query = "From when the generative ai gets popular ?"

result = vector_db.similarity_search(query)

In [14]:
print(result[0].page_content)

Generative AI adoption[edit]
Main article: AI boom
AI generated images have become much more advanced.
In March 2020, the release of 15.ai, a free web application created by an anonymous MIT researcher that could generate convincing character voices using minimal training data, was one of the earliest publicly available uses for generative AI.[34] The platform is credited as the first mainstream service for audio deepfakes.[35][36]
In 2021, DALL-E, a closed-source transformer-based generative model developed by OpenAI, drew widespread attention to text-to-image generation.[37]
Other projects, including open-source approaches such as VQGAN+CLIP and DALL·E Mini (later renamed Craiyon), made similar systems more accessible to the public.[38]
Dream by Wombo was released at the end of 2021,[39] followed by the releases of Midjourney and Stable Diffusion in 2022.[40][41]


In [15]:
llm = ChatOpenAI(model = "gpt-5.4-nano")

In [21]:
## retrieval chain and document chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    ''' 
        Answer the following question based only on the provided context:
        <context>
        {context}
        </context>
    '''
)

document_chain = create_stuff_documents_chain(llm, prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=' \n        Answer the following question based only on the provided context:\n        <context>\n        {context}\n        </context>\n    '), additional_kwargs={})])
| ChatOpenAI(output_version=None, profile={'name': 'GPT-5.4 nano', 'release_date': '2026-03-17', 'last_updated': '2026-03-17', 'open_weights': False, 'max_input_tokens': 400000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calli

In [22]:
from langchain_core.documents import Document
document_chain.invoke({
    "input": "From when the generative ai gets popular ?",
    "context":[Document(page_content="In March 2020, the release of 15.ai, a free web application created by an anonymous MIT researcher that could generate convincing character voices using minimal training data, was one of the earliest publicly available uses for generative AI.[34] The platform is credited as the first mainstream service for audio deepfakes.")]
})



'In March 2020, the release of **15.ai**—a free web application by an anonymous MIT researcher that could generate convincing character voices with minimal training data—was one of the earliest publicly available uses of generative AI, and it is credited as the **first mainstream service for audio deepfakes**.'

In [23]:
retriever = vector_db.as_retriever()

from langchain_classic.chains import create_retrieval_chain

retrival_chain = create_retrieval_chain(retriever, document_chain)


In [25]:
retrival_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x12052d720>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=' \n        Answer the following question based only on the provided context:\n        <context>\n        {context}\n        </context>\n    '), additional_k

In [26]:
response = retrival_chain.invoke({"input": "From when the generative ai gets popular? "})

In [29]:
print(response['answer'])

The context describes generative AI adoption starting with early public tools like **15.ai** (released in **March 2020**), which could generate convincing character voices and is credited as the first mainstream service for **audio deepfakes**. It then notes that in **2021**, **DALL-E** (OpenAI) brought widespread attention to **text-to-image** generation, and that subsequent open-source projects (e.g., **VQGAN+CLIP**, **DALL·E Mini/Craiyon**) made similar systems more accessible. Later, **Dream by Wombo** (late **2021**), and then **Midjourney** and **Stable Diffusion** (**2022**) were released, marking further adoption and accessibility of generative AI tools.


In [28]:
response

{'input': 'From when the generative ai gets popular? ',
 'context': [Document(id='2d298c00-9480-4de0-90dc-d0e1e40e19aa', metadata={'source': 'https://en.wikipedia.org/wiki/Generative_AI', 'title': 'Generative AI - Wikipedia', 'language': 'en'}, page_content='Generative AI adoption[edit]\nMain article: AI boom\nAI generated images have become much more advanced.\nIn March 2020, the release of 15.ai, a free web application created by an anonymous MIT researcher that could generate convincing character voices using minimal training data, was one of the earliest publicly available uses for generative AI.[34] The platform is credited as the first mainstream service for audio deepfakes.[35][36]\nIn 2021, DALL-E, a closed-source transformer-based generative model developed by OpenAI, drew widespread attention to text-to-image generation.[37]\nOther projects, including open-source approaches such as VQGAN+CLIP and DALL·E Mini (later renamed Craiyon), made similar systems more accessible to the

In [30]:
response['context']

[Document(id='2d298c00-9480-4de0-90dc-d0e1e40e19aa', metadata={'source': 'https://en.wikipedia.org/wiki/Generative_AI', 'title': 'Generative AI - Wikipedia', 'language': 'en'}, page_content='Generative AI adoption[edit]\nMain article: AI boom\nAI generated images have become much more advanced.\nIn March 2020, the release of 15.ai, a free web application created by an anonymous MIT researcher that could generate convincing character voices using minimal training data, was one of the earliest publicly available uses for generative AI.[34] The platform is credited as the first mainstream service for audio deepfakes.[35][36]\nIn 2021, DALL-E, a closed-source transformer-based generative model developed by OpenAI, drew widespread attention to text-to-image generation.[37]\nOther projects, including open-source approaches such as VQGAN+CLIP and DALL·E Mini (later renamed Craiyon), made similar systems more accessible to the public.[38]\nDream by Wombo was released at the end of 2021,[39] fo